# Synthetic data from knowledge

Create a synthetic dataset that starts from comics conversations and raw
knowledge and produces conversations that match the style of the comic, while
answering the questions.

In [ ]:
import time

input_convos = "output/dataset/dspy-dataset.csv"
input_knowledge = "output/schede/*.json"
example_output = f"output/synthetic/examples/{time.strftime('%Y-%m-%d-%H-%M')}.json"
qa_output_dir = "output/synthetic/qa"

In [54]:
import json
from glob import glob
from pathlib import Path
from dataclasses import dataclass


@dataclass
class KnowledgeDoc:
    title: str
    text: str
    input_page: str
    text_path: Path

    def __str__(self):
        return f"# {self.title}\n\n{self.text}\n"


def parse_knowledge_file(file_path: Path) -> KnowledgeDoc:
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)
    return KnowledgeDoc(
        title=data['title'],
        text=data['text'],
        input_page=data['input_page'],
        text_path=file_path,
    )


knowledge_docs = [
    parse_knowledge_file(Path(file))
    for file in glob(input_knowledge)
]
len(knowledge_docs)

170

In [21]:
import pandas as pd

df = pd.read_csv(input_convos)
df.head()

,issue,input_pages,characters,conversations
0,PKNA #0,"[""input/pkna/pkna-0/pkna-0-029.jpg"", ""../in...","[""other"", ""pk"", ""uno""]","[{""role"": ""human"", ""content"": ""Hai paura di af..."
1,PKNA #0,"[""input/pkna/pkna-0/pkna-0-046.jpg"", ""../in...","[""other"", ""pk"", ""uno""]","[{""role"": ""human"", ""content"": ""Ci sei, uno? Pu..."
2,PKNA #0,"[""input/pkna/pkna-0/pkna-0-057.jpg"", ""../in...","[""other"", ""pk"", ""uno""]","[{""role"": ""human"", ""content"": ""Intervento prov..."
3,PKNA #0,"[""input/pkna/pkna-0/pkna-0-069.jpg"", ""../in...","[""pk"", ""uno""]","[{""role"": ""human"", ""content"": ""Sempre su...\nE..."
4,PKNA #0/2,"[""input/pkna/pkna-0-2/PK0-2 005.jpg"", ""../i...","[""pk"", ""uno""]","[{""role"": ""human"", ""content"": ""Buongiorno, uno..."


In [22]:
import json


@dataclass
class ExampleDialogue:
    lines: list[str]
    input_pages: list[str]


def parse_conversation(conversation: str, input_pages: str) -> ExampleDialogue:
    """Parse a conversation string into a list of messages."""
    return ExampleDialogue(
        lines=[
            line['content']
            for line in json.loads(conversation)
        ],
        input_pages=[p for p in json.loads(input_pages)],
    )


example_conversations = [
    parse_conversation(c, ip)
    for c, ip in zip(df['conversations'].tolist(), df['input_pages'].tolist())
]
example_conversations[0]

ExampleDialogue(lines=['Hai paura di affrontarmi? Fatti vedere!\nVeramente, mi stai già guardando!', 'Io sono questo edificio!', 'Sglurg!', 'Ma se proprio hai bisogno di un volto a cui rivolgerti ... ecco!', 'Glom! Penso di averne viste abbastanza, per una sola notte!\nFacciamoci coraggio!\nNon so cosa tu sia, ma hai di fronte Paperi-Nik!', 'Piacere! Io sono uno!', 'Uno, eh? Dove hai lasciato gli altri?', 'Groan! Che battutaccia! È terribile!', 'Immagino che ti aspetti spiegazioni! Mettiti comodo!', 'Molto Gentile!', 'Dunque sei un computer?', "Un'intelligenza artificiale, prego!\nPer essere precisi, sono la più potente, versatile e sbalorditiva intelligenza artificiale mai esistita su questo pianeta!\nEverett Ducklair mi ha programmato per avere qualcuno al suo livello intellettuale con cui parlare!", "Ti ha dato anche la sua stessa modestia, direi. Ih! Ih!\nDov'è adesso il tuo amico cervellone, uno? Pare che abbia levato le tende!\nSigh! È una storia piuttosto complicata!\nRaccontart

In [23]:
def dialogue_to_qa(dialogue: ExampleDialogue) -> list[tuple[str, str]]:
    """Convert a dialogue into a list of question-answer pairs."""
    qa_pairs = []
    for i in range(0, len(dialogue.lines) - 1, 2):
        candidate = dialogue.lines[i]
        if "?" not in candidate:
            continue
        qa_pairs.append((candidate, dialogue.lines[i + 1]))
    return qa_pairs

dialogue_to_qa(example_conversations[0])[:5]

[('Hai paura di affrontarmi? Fatti vedere!\nVeramente, mi stai già guardando!',
  'Io sono questo edificio!'),
 ('Uno, eh? Dove hai lasciato gli altri?',
  'Groan! Che battutaccia! È terribile!'),
 ('Dunque sei un computer?',
  "Un'intelligenza artificiale, prego!\nPer essere precisi, sono la più potente, versatile e sbalorditiva intelligenza artificiale mai esistita su questo pianeta!\nEverett Ducklair mi ha programmato per avere qualcuno al suo livello intellettuale con cui parlare!"),
 ("Ti ha dato anche la sua stessa modestia, direi. Ih! Ih!\nDov'è adesso il tuo amico cervellone, uno? Pare che abbia levato le tende!\nSigh! È una storia piuttosto complicata!\nRaccontartela per immagini sarà più facile!",
  'Veramente è una simulazione a ipercompressione di dati. Ma lasciamo perdere!'),
 ('E secondo te sono giocattoli? Tsk!\nQuesti marchingegni sembrano molto superiori ai miei soliti trucchi!\nScommetto che funzionerebbero anche contro quei fiammiferoni blu che ho incontrato ieri!',


In [43]:
example_qas: list[ExampleDialogue] = []
for conversation in example_conversations:
    qas = dialogue_to_qa(conversation)
    for qa in qas:
        example_qas.append(ExampleDialogue(
            lines=[qa[0], qa[1]],
            input_pages=conversation.input_pages,
        ))

len(example_qas)

397

In [29]:
def load_bio(name: str) -> str:
    with open(f"input/bios/{name}.md", encoding="utf-8") as f:
        return f.read().strip()

pk_bio = load_bio("paperinik")
uno_bio = load_bio("uno")

In [25]:
import os

from dotenv import load_dotenv
import dspy


load_dotenv()

lm = dspy.LM(
    model='vertex_ai/gemini-2.5-flash',
    vertex_credentials=os.getenv('VERTEX_AI_CREDS'),
    temperature=1.0,
    top_p=0.95,
    top_k=64,
    max_tokens=65535,
)
dspy.configure(lm=lm, track_usage=True)

## Q/A generation

Create a specialized module for Q/A only.

In [38]:
import dspy
from pydantic import BaseModel


class QuestionAnswerPair(BaseModel):
    """A question and its corresponding answer."""
    question: str
    answer: str


class QAGenerator(dspy.Signature):
    """Generate question and answer pairs based on knowledge in the input
    content, while matching the styles of the provided examples."""

    knowledge: str = dspy.InputField(
        desc="Knowledge document to use for generating responses.",
    )
    examples: list[QuestionAnswerPair] = dspy.InputField(
        desc="Example question-answer pairs you must match the style of.",
        max_length=10,
    )
    question_speaker_bio: str = dspy.InputField(
        desc="Bio of the speaker to use for the generated question.",
    )
    answer_speaker_bio: str = dspy.InputField(
        desc="Bio of the speaker to use for the generated answer.",
    )
    generated: list[QuestionAnswerPair] = dspy.OutputField(
        desc="A list of question-answer pairs.",
        min_length=5,
        max_length=10,
    )


class SyntheticQAModule(dspy.Module):
    """A module for generating synthetic responses based on input text and
    knowledge documents."""

    def __init__(self):
        super().__init__()
        self.generator = dspy.ChainOfThought(QAGenerator)

    def forward(self,
                knowledge_doc: str,
                model_qas: list[QuestionAnswerPair],
                speakers_bio: tuple[str, str],
                ) -> dspy.Prediction:
        pred = self.generator(
            knowledge=knowledge_doc,
            examples=model_qas,
            question_speaker_bio=speakers_bio[0],
            answer_speaker_bio=speakers_bio[1],
        ).generated
        return dspy.Prediction(qa_pairs=pred)


generator = SyntheticQAModule()

In [ ]:
import random

examples = example_qas.copy()
random.Random(42).shuffle(examples)
examples = examples[:5]

prediction = generator(
    knowledge_doc=str(knowledge_docs[0]),
    model_qas=[
        QuestionAnswerPair(question=qa.lines[0], answer=qa.lines[1])
        for qa in examples
    ],
    speakers_bio=(pk_bio, uno_bio),
)
prediction.qa_pairs

[QuestionAnswerPair(question="Uno, questo 'TUTTO PATEMI!' è un caos totale! Due Perceval, Brenda che si scopre 'Liza'... Mi spieghi come si fa a uscire da un simile pasticcio? E soprattutto, 'inconsapevoli'... ma di cosa, esattamente?", answer="La situazione è un esempio estremo di 'telenovela logica', Paperinik. L'inconsapevolezza diffusa è il motore primario del conflitto. Per uscirne, basterebbe un riscontro di identità e un test del DNA. Elementare, per chi non è 'inconsapevole'."),
 QuestionAnswerPair(question='Quella Brenda, poi! Giura vendetta per così poco? Mi sa che ha un bel caratterino... Che piani ha in mente, Uno? E quanto è pericolosa?', answer="Brenda Fellow è altamente motivata dalla gelosia e dall'ambizione. I suoi piani sono improntati alla destabilizzazione sociale e legale. Pericolosità: media, ma con potenziale di escalation drammatica. La sua capacità di generare 'colpi di scena' è notevole."),
 QuestionAnswerPair(question='Il gemello povero di Perceval! Dalla cul

In [41]:
prediction.get_lm_usage()

{'vertex_ai/gemini-2.5-flash': {'completion_tokens': 5378,
  'prompt_tokens': 2300,
  'total_tokens': 7678,
  'completion_tokens_details': {'accepted_prediction_tokens': None,
   'audio_tokens': None,
   'reasoning_tokens': 4134,
   'rejected_prediction_tokens': None,
   'text_tokens': 1244},
  'prompt_tokens_details': {'audio_tokens': None,
   'cached_tokens': None,
   'text_tokens': 2300,
   'image_tokens': None}}}

In [ ]:
import json


def save_example(
        pred: dspy.Prediction,
        knowledge_doc_path: str, 
        examples: list[ExampleDialogue],
        character_names: tuple[str, str],
        out_path: str,
) -> None:
    examples_paths = {
        page
        for example in examples
        for page in example.input_pages
    }
    data = {
        "qa_pairs": [
            {"question": qa.question, "answer": qa.answer}
            for qa in pred.qa_pairs
        ],
        "characters": character_names,
        "knowledge_doc": knowledge_doc_path,
        "example_pages": list(examples_paths),
        "example_qa_pairs": [
            {"question": qa.lines[0], "answer": qa.lines[1]}
            for qa in examples
        ],
        "lm_usage": pred.get_lm_usage(),
    }

    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


save_example(
    prediction,
    knowledge_docs[0].input_page,
    examples,
    ("paperinik", "uno"),
    example_output,
)

In [56]:
# Run the generation
import random
from tqdm.notebook import tqdm


def generate_synthetic_qa(
        knowledge_docs: list[KnowledgeDoc],
        example_qas: list[ExampleDialogue],
        character_bios: tuple[str, str],
        character_names: tuple[str, str],
) -> None:
    """Generate synthetic question-answer pairs based on the provided
    knowledge documents and example QAs."""

    rnd = random.Random(42)
    def reset_examples():
        """Get a shuffled list of examples."""
        examples = example_qas.copy()
        rnd.shuffle(examples)
        return examples
    curr_examples = reset_examples()

    def next_examples(n: int) -> list[ExampleDialogue]:
        """Get the next set of examples."""
        nonlocal curr_examples
        if len(curr_examples) < n:
            curr_examples = reset_examples()
        res, curr_examples = curr_examples[:n], curr_examples[n:]
        return res

    generator = SyntheticQAModule()

    for knowledge_doc in tqdm(knowledge_docs, desc="Knowledge Documents"):
        # Generate 4 predictions for each knowledge document
        for i in range(4):
            examples = next_examples(5)
            out_path = f"{qa_output_dir}/{knowledge_doc.text_path.stem}-example-{i}.json"
            if Path(out_path).exists():
                continue

            prediction = generator(
                knowledge_doc=str(knowledge_doc),
                model_qas=[
                    QuestionAnswerPair(question=qa.lines[0], answer=qa.lines[1])
                    for qa in examples
                ],
                speakers_bio=character_bios,
            )
            save_example(
                prediction,
                knowledge_doc.input_page,
                examples,
                character_names,
                out_path,
            )

In [58]:
generate_synthetic_qa(
    knowledge_docs,
    example_qas,
    (pk_bio, uno_bio),
    ("paperinik", "uno"),
)

Knowledge Documents:   0%|          | 0/170 [00:00<?, ?it/s]

KeyboardInterrupt: 

# OLD

In [ ]:
import dspy
from pydantic import BaseModel, Field


class DialogueLine(BaseModel):
    """A line in a conversation, spoken by a specific speaker."""
    content: str


class Conversation(BaseModel):
    """A conversation consisting of multiple dialogue lines. Lines always
    alternate between two speakers."""

    speaker1: str
    speaker2: str
    lines: list[DialogueLine] = Field(default_factory=list, max_length=6)


class QuestionAnswerPair(BaseModel):
    """A question and its corresponding answer."""
    question: str
    answer: str


class QAGenerator(dspy.Signature):
    """Generate 10 questions and answers pairs based on knowledge in the input
    content, while matching the style of the provided conversation.

    Each question should be self-contained and not a continuation of the
    previous one. The answers should be concise and directly address the
    questions asked."""

    knowledge: str = dspy.InputField(
        desc="Knowledge document to use for generating responses.")
    model_conversation: Conversation = dspy.InputField(
        desc="A sample conversation you must match the style of."
    )
    qa_pairs: list[QuestionAnswerPair] = dspy.OutputField(
        desc="A list of question-answer pairs."
    )


class MultiTurnGenerator(dspy.Signature):
    """Generate back and forth conversations based on knowledge in the input
    content, while matching the style of the provided conversation."""

    knowledge: str = dspy.InputField(
        desc="Knowledge document to use for generating responses.")
    model_conversation: Conversation = dspy.InputField(
        desc="A sample conversation you want to match the style of."
    )
    conversations: list[Conversation] = dspy.OutputField(
        desc="A list of conversations to generate responses for.",
        max_length=10,
    )


class SyntheticModule(dspy.Module):
    """A module for generating synthetic responses based on input text and
    knowledge documents."""

    def __init__(self):
        super().__init__()
        self.multiturn_gen = dspy.ChainOfThought(MultiTurnGenerator)
        self.qa_gen = dspy.ChainOfThought(QAGenerator)

    def forward(self, knowledge_doc: str, model_convo: Conversation) -> dspy.Prediction:
        multi_turn = self.multiturn_gen(
            knowledge=knowledge_doc,
            model_conversation=model_convo,
        )
        qa = self.qa_gen(
            knowledge=knowledge_doc,
            model_conversation=model_convo,
        )
        return dspy.Prediction(
            multi_turn=[
                [line for line in convo.lines]
                for convo in multi_turn.conversations
            ],
            qa_pairs=[
                [pair.question, pair.answer]
                for pair in qa.qa_pairs
            ]
        )


generator = SyntheticModule()

In [21]:
def example_to_conversation(example: list[str]) -> Conversation:
    """Convert a list of dialogue lines into a Conversation object."""
    return Conversation(
        speaker1="paperinik",
        speaker2="uno",
        lines=[DialogueLine(content=line) for line in example]
    )

In [ ]:
import os

from dotenv import load_dotenv
import dspy


load_dotenv()

lm = dspy.LM(
    model='vertex_ai/gemini-2.5-pro',
    vertex_credentials=os.getenv('VERTEX_AI_CREDS'),
    temperature=1.0,
    top_p=0.95,
    top_k=64,
    max_tokens=65535,
)
dspy.configure(lm=lm, track_usage=True)

In [26]:
example_pred = generator(knowledge_doc=knowledge_docs[0],
                         model_convo=example_to_conversation(
                             example_conversations[0]),
                         )
example_pred

Prediction(
    multi_turn=[[DialogueLine(content="Quella smorfiosa di Liza Laker è la nuova direttrice della Shirt! A me, che respiro moda da quando sono nata! È un'ingiustizia!"), DialogueLine(content="Mmh? Scusa, cara, dicevi? Stavo... ehm... ammirando la tappezzeria. Molto... d'impatto."), DialogueLine(content='Stai guardando lei! Non negarlo! I tuoi occhi brillano come fari! Aargh! Non ti sarai mica innamorato di quella nullità inconsapevole?'), DialogueLine(content="Io? Ma no! Certo, è bella, giovane... e inconsapevole... Credo... credo di doverla conoscere meglio. Per affari, s'intende!")], [DialogueLine(content='Paparino! Davvero? Io, direttrice della Shirt? Ma è una responsabilità enorme! Non so se sono pronta!'), DialogueLine(content='Sciocchezze, figliola mia! Hai il mio stesso fiuto per gli affari, anche se lo ignori! Sei una Laker, dopotutto!'), DialogueLine(content='Oh, grazie! Sarà bellissimo lavorare con Brenda! So che sarà felicissima per me, è così buona e gentile!'),

In [ ]:
def save_example(pred, out_path):
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump({
            "multi_turn": [
                [line.content for line in convo]
                for convo in pred.multi_turn
            ],
            "qa_pairs": pred.qa_pairs,
        }, f, ensure_ascii=False, indent=2)


save_example(example_pred, example_output)